In [ ]:
# Part 1: Setup

# importing libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

In [ ]:
# check versions
print(f"pandas version {pd.__version__}")
print(f"numpy version {np.__version__}")
# print(f"matplotlib version {plt.__version__}")
print(f"seaborn version {sns.__version__}")

In [ ]:
HIGH_VOLUME_ROLES = {"fraud_investigator", "contact_centre_agent"}
EXCESSIVE_ACCESS_MULTIPLIER = 3

In [ ]:
# part 1.2 standardize text
def standardize_text(value: str) -> str:
    '''
        Cleans up messy categorical values

        Handles whitespace, runs of spaces, newlines,
        hyphens and repeated underscores.
    '''
    clean_value = "_".join(
        value.lower()
        .replace("-", " ")
        .replace("_", " ")
        .split()
    )

    return clean_value

In [ ]:
# Part 2: NumPy
adviser_counts = np.array([147, 29, 35, 31, 28, 33, 26, 40, 110, 32])

# mean
mean = np.mean(adviser_counts)

# median
median = np.median(adviser_counts)

# percentile
percentile_90th = np.percentile(adviser_counts, 90)

print(f"mean {mean}, median {median}, 90th percentile {percentile_90th:.2f}")

In [ ]:
# 2.2 Calculate every adviser’s ratio to the median in one line, without a loop.
adviser_ratio = np.array(adviser_counts) / median


print(f"{adviser_ratio}")

# 2.3 Create a boolean array showing which ratios exceed EXCESSIVE_ACCESS_MULTIPLIER
# and use .sum() on it to count how many do. Hint: when summed, True counts as 1 and False as 0.

is_excessive_access = np.array(adviser_ratio) > EXCESSIVE_ACCESS_MULTIPLIER
print(f"Excessive Access: {sum(is_excessive_access)} ")

In [ ]:
# part 3

employee_week_data = {
    "employee_id": ["E00417", "E00588", "E00612", "E00655", "E00701", "E00733", "E00790",
                    "E00812", "E00845", "E00861", "E00877", "E00902", "E00915", "E00928",
                    "E00731", "E00744", "E00756", "E00768", "E01044", "E01077"],
    "role": ["branch_adviser", "branch_adviser", "branch_adviser", "branch_adviser",
             "Branch Adviser", "branch_adviser", "branch_adviser", "branch_adviser",
             "contact_centre_agent", "contact_centre_agent", "contact_centre_agent",
             "contact_centre_agent", "contact_centre_agent", "contact_centre_agent",
             "fraud_investigator", "fraud-investigator", "fraud_investigator",
             "fraud_investigator", "branch_adviser", "branch_adviser"],
    "location": ["leeds_briggate", "leeds_briggate", "york_coney_street", "york_coney_street",
                 "hull_whitefriargate", "hull_whitefriargate", "leeds_briggate",
                 "york_coney_street", "leeds_contact_centre", "leeds_contact_centre",
                 "leeds_contact_centre", "leeds_contact_centre", "leeds_contact_centre",
                 "leeds_contact_centre", "head_office", "head_office", "head_office",
                 "head_office", "hull_whitefriargate", "york_coney_street"],
    "unique_customers": [147, 29, 35, 31, 28, 33, 26, 40, 205, 218, 196, 250, 211, 189,
                         640, 175, 182, 168, 110, 32],
    "after_hours_events": [11, 0, 1, 0, 2, 0, 1, 3, 4, 6, 2, 9, 5, 3, 22, 8, 10, 7, 0, 0],
}

employee_week_df = pd.DataFrame(employee_week_data)

In [ ]:
# 3.1 first 5 rows
employee_week_df.head()

In [ ]:
# 3.2 the shape
employee_week_df.shape

In [ ]:
#3.3
employee_week_df.info()

In [ ]:
# 3.4
employee_week_df.describe()

In [ ]:
# employee_id: str
# role : str
# location: str
# unique_customers: int64
# after_hours_events: int 64


# what describe says about unique_customers
# there are 20 rows with a mean of 147.25 and median of 157.5 indicating that the data is
# left skewed.

In [ ]:
# part 4
median_by_role = (
    employee_week_df.groupby("role")["unique_customers"].median()
    )

print(median_by_role)

# Got 5 groups because of inconsistent data format

# create a clean role column
employee_week_df["clean_role"] = employee_week_df["role"].apply(standardize_text)

# median on clean role
peer_medians = (
    employee_week_df.groupby("clean_role")["unique_customers"].median()
)

print("_" * 40)
print(peer_medians)

# peer_median column
employee_week_df["peer_median"] = employee_week_df["clean_role"].map(peer_medians)

# ratio to peer column
employee_week_df["ratio_to_peer"] = employee_week_df["unique_customers"] / employee_week_df["peer_median"]

display(employee_week_df.head(2))

In [ ]:
# filter
alerted_df = (employee_week_df[(
    employee_week_df["ratio_to_peer"] > EXCESSIVE_ACCESS_MULTIPLIER) 
    & ~employee_week_df["clean_role"].isin(HIGH_VOLUME_ROLES)]
    )

display(alerted_df[["employee_id", "clean_role", "ratio_to_peer"]])

In [ ]:
# part 5
# I don't understand the charting yet, do I need to always do a copy and paste for them?
sorted_df = employee_week_df.sort_values("unique_customers", ascending=False)
fig, ax = plt.subplots(figsize=(10, 4))
ax.bar(sorted_df["employee_id"], sorted_df["unique_customers"])
ax.set_title("Unique customers access by employee (one week)")
ax.set_xlabel("Employee")
ax.set_ylabel("Number of unique customers accessed")
ax.tick_params(axis="x", rotation=90)
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(10, 4))
sns.boxplot(data=employee_week_df, x="clean_role", y="unique_customers", ax=ax)
ax.set_title("Unique customers access by employee (one week)")
ax.set_xlabel("Role")
ax.set_ylabel("Unique customers accessed")
plt.show()

In [ ]:
sorted_median = peer_medians.sort_values(ascending=False)

fig, ax = plt.subplots(figsize=(10, 4))
ax.bar(sorted_median.index, sorted_median.values)
ax.set_title("Peer Median by Role")
ax.set_xlabel("Index")
ax.set_ylabel("values")
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(10, 4))
sns.histplot(data=employee_week_df, x="after_hours_events", hue=("clean_role"), ax=ax)
ax.set_title("Distribution of after hours event")
ax.set_xlabel("After hours access events")
plt.show()